## Toxic Comment Classification

This project detects different types of toxicity in online comments (Wikipedia talk page edits). A comment can be toxic, severely toxic, obscene, a threat, an insult, or contain identity-based hate, and it can have several of these labels at once, so this is a multi-label classification problem.

## Approach
1. Load the data and explore label frequencies
2. Convert comment text into numeric features with TF-IDF
3. Train one logistic regression per label and evaluate the mean AUC on a validation split
4. Generate probabilities and create the submission file
5. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge
!unzip -oq jigsaw-toxic-comment-classification-challenge.zip
!unzip -oq train.csv.zip
!unzip -oq test.csv.zip
!unzip -oq sample_submission.csv.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 1.5 MB/s eta 0:00:00
100% 52.6M/52.6M [00:02<00:00, 21.6MB/s]



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [4]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            153164 non-null  object
 1   comment_text  153164 non-null  object
dtypes: object(2)
memory usage: 2.3+ MB


In [5]:
train.sample(1)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
128564,af9888f7aa656b9b,I agree we should wait for citations.,0,0,0,0,0,0


In [6]:
test.sample(1)

,id,comment_text
71915,77d17d9d568ec208,So many people are complaining about it... mus...


In [7]:
train.shape, test.shape

((159571, 8), (153164, 2))

In [8]:
train.columns.tolist()

['id',
 'comment_text',
 'toxic',
 'severe_toxic',
 'obscene',
 'threat',
 'insult',
 'identity_hate']

In [9]:
train[["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]].mean()

,0
toxic,0.095844
severe_toxic,0.009996
obscene,0.052948
threat,0.002996
insult,0.049364
identity_hate,0.008805


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=30000, stop_words="english", sublinear_tf=True)
x = vectorizer.fit_transform(train["comment_text"])

In [11]:
y = train[["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]]

In [12]:
x.shape, y.shape

((159571, 30000), (159571, 6))

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

model = OneVsRestClassifier(LogisticRegression(C=4, max_iter=1000), n_jobs=-1)
model.fit(x_train, y_train)

proba = model.predict_proba(x_val)
print("Validation mean AUC:", round(roc_auc_score(y_val, proba, average="macro"), 4))

Validation mean AUC: 0.9788


In [14]:
model.fit(x, y)

x_test = vectorizer.transform(test["comment_text"])
proba_test = model.predict_proba(x_test)

submission = pd.DataFrame(proba_test, columns=["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"])
submission.insert(0, "id", test["id"])
submission.to_csv("submission.csv", index=False)

In [15]:
!kaggle competitions submit -c jigsaw-toxic-comment-classification-challenge -f submission.csv -m "TF-IDF + OneVsRest LogisticRegression"

100% 21.0M/21.0M [00:02<00:00, 8.76MB/s]
99 submissions remaining today.
Successfully submitted to Toxic Comment Classification Challenge

In [16]:
import pickle, os
pickle.dump(model, open("toxic_model.pkl", "wb"))
pickle.dump(vectorizer, open("toxic_vectorizer.pkl", "wb"))
print("model MB:", round(os.path.getsize("toxic_model.pkl") / 1e6, 2))
print("vectorizer MB:", round(os.path.getsize("toxic_vectorizer.pkl") / 1e6, 2))

model MB: 1.44
vectorizer MB: 1.11
